#### To try out stuff for Modeling and Data Change and See how it works

In [174]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

from bike_rental.defs.assets.helper import (
    add_lag_features,
    add_rolling_features,
    add_time_based_features,
)

# go to project root (parent of week-3)
project_root = Path.cwd().resolve().parent

# add src to Python path
sys.path.append(str(project_root / "src"))

In [175]:
from bike_rental.defs.resources.project_config import ProjectConfig

cfgs = ProjectConfig()
cfgs.curated_path

'data/raw/curated_rental_dataset.csv'

In [176]:
data = pd.read_csv(f"../{cfgs.curated_path}", parse_dates=["datetime"])
data = data.sort_values("datetime")

In [177]:
data.head()

,location_id,datetime,count_rentals,count_pickups,weekday,year,month,day,quarter,hour,...,total_count,temperature_c,perceived_temperature_c,humidity,windspeed_kmh,conditions_clear,conditions_clouds,conditions_heavy_rain,conditions_light_rain,is_holiday
0,2,2011-01-01,0.0,1.0,5,2011,1,1,1,0,...,1.0,3.3,3.0,81.0,0.0,1.0,0.0,0.0,0.0,0
9,16,2011-01-01,1.0,0.0,5,2011,1,1,1,0,...,1.0,3.3,3.0,81.0,0.0,1.0,0.0,0.0,0.0,0
8,14,2011-01-01,0.0,1.0,5,2011,1,1,1,0,...,1.0,3.3,3.0,81.0,0.0,1.0,0.0,0.0,0.0,0
7,13,2011-01-01,2.0,0.0,5,2011,1,1,1,0,...,2.0,3.3,3.0,81.0,0.0,1.0,0.0,0.0,0.0,0
6,12,2011-01-01,2.0,0.0,5,2011,1,1,1,0,...,2.0,3.3,3.0,81.0,0.0,1.0,0.0,0.0,0.0,0


In [178]:
data.shape

(368412, 22)

In [179]:
data.isna().sum()

location_id                   0
datetime                      0
count_rentals                 0
count_pickups                 0
weekday                       0
year                          0
month                         0
day                           0
quarter                       0
hour                          0
is_month_start                0
is_month_end                  0
total_count                   0
temperature_c              3465
perceived_temperature_c    3465
humidity                   3465
windspeed_kmh              3465
conditions_clear           3465
conditions_clouds          3465
conditions_heavy_rain      3465
conditions_light_rain      3465
is_holiday                    0
dtype: int64

In [180]:
data.isna().sum().sum()

np.int64(27720)

In [181]:
data = data.drop_duplicates()
data = data.dropna()
data = data.sort_values("datetime")
data.reset_index(drop=True, inplace=True)
data.shape

(364947, 22)

In [182]:
data.columns

Index(['location_id', 'datetime', 'count_rentals', 'count_pickups', 'weekday',
       'year', 'month', 'day', 'quarter', 'hour', 'is_month_start',
       'is_month_end', 'total_count', 'temperature_c',
       'perceived_temperature_c', 'humidity', 'windspeed_kmh',
       'conditions_clear', 'conditions_clouds', 'conditions_heavy_rain',
       'conditions_light_rain', 'is_holiday'],
      dtype='str')

In [183]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 364947 entries, 0 to 364946
Data columns (total 22 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   location_id              364947 non-null  int64         
 1   datetime                 364947 non-null  datetime64[us]
 2   count_rentals            364947 non-null  float64       
 3   count_pickups            364947 non-null  float64       
 4   weekday                  364947 non-null  int64         
 5   year                     364947 non-null  int64         
 6   month                    364947 non-null  int64         
 7   day                      364947 non-null  int64         
 8   quarter                  364947 non-null  int64         
 9   hour                     364947 non-null  int64         
 10  is_month_start           364947 non-null  int64         
 11  is_month_end             364947 non-null  int64         
 12  total_count              36

In [184]:
def linear_regression_model(X_train, y_train):
    """Train a Linear Regression model on the training data."""
    model = LinearRegression()
    model.fit(X_train, y_train)
    return model


def random_forest_model(X_train, y_train):
    """Train a Random Forest regression model on the training data."""
    model = RandomForestRegressor(
        n_estimators=200, max_depth=None, random_state=42, n_jobs=-1
    )
    model.fit(X_train, y_train)
    return model


def xgboost_model(X_train, y_train):
    """Train an XGBoost regression model on the training data."""
    model = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
    )
    model.fit(X_train, y_train)
    return model

In [185]:
def metrics(model, X_test, y_test):
    """Evaluate the XGBoost model on the test data."""
    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    return {"mae": mae, "rmse": rmse}

In [186]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 364947 entries, 0 to 364946
Data columns (total 22 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   location_id              364947 non-null  int64         
 1   datetime                 364947 non-null  datetime64[us]
 2   count_rentals            364947 non-null  float64       
 3   count_pickups            364947 non-null  float64       
 4   weekday                  364947 non-null  int64         
 5   year                     364947 non-null  int64         
 6   month                    364947 non-null  int64         
 7   day                      364947 non-null  int64         
 8   quarter                  364947 non-null  int64         
 9   hour                     364947 non-null  int64         
 10  is_month_start           364947 non-null  int64         
 11  is_month_end             364947 non-null  int64         
 12  total_count              36

In [205]:
def splitting_data(daily_data: pd.DataFrame, features: list):
    """Split the data into training and testing sets based on a 70-30 split."""
    split_index = int(len(daily_data) * 0.7)

    train = daily_data.iloc[:split_index]
    test = daily_data.iloc[split_index:]

    X_train = train[features]
    y_train = train["total_count"]

    X_test = test[features]
    y_test = test["total_count"]
    return X_train, y_train, X_test, y_test

# Different Feature Sets, and Trial

In [204]:
FEATURE_SETS = {
    "weather_only": [
        "temperature_c",
        "perceived_temperature_c",
        "humidity",
        "windspeed_kmh",
        "conditions_clear",
        "conditions_clouds",
        "conditions_heavy_rain",
        "conditions_light_rain",
    ],
    "feature_set_1": [
        "is_month_start",
        "is_month_end",
        "temperature_c",
        "perceived_temperature_c",
        "humidity",
        "windspeed_kmh",
        "conditions_clear",
        "conditions_clouds",
        "conditions_heavy_rain",
        "conditions_light_rain",
        "is_holiday",
    ],
    "feature_set_2": [
        "dayofweek",
        "year",
        "month",
        "day",
        "quarter",
        "hour",
        "is_month_start",
        "is_month_end",
        "is_weekend",
    ],
    "feature_set_3": [
        "dayofweek",
        "year",
        "month",
        "day",
        "quarter",
        "hour",
        "is_month_start",
        "is_month_end",
        "is_weekend",
        "hour_sin",
        "hour_cos",
    ],
    "feature_set_4": [
        "dayofweek",
        "year",
        "month",
        "day",
        "quarter",
        "hour",
        "is_month_start",
        "is_month_end",
        "is_weekend",
        "hour_sin",
        "hour_cos",
        "total_count_lag_1",
        "total_count_lag_24",
        "total_count_lag_168",
    ],
    "feature_set_5": [
        "dayofweek",
        "year",
        "month",
        "day",
        "quarter",
        "hour",
        "is_month_start",
        "is_month_end",
        "is_weekend",
        "hour_sin",
        "hour_cos",
        "total_count_lag_1",
        "total_count_lag_24",
        "total_count_lag_168",
        "total_count_rolling_mean_24",
        "total_count_rolling_mean_168",
        "total_count_rolling_std_24",
        "total_count_rolling_std_168",
    ],
}


TARGET = "total_count"

In [203]:
# Since We are not using the Location,
# First I will aggregate the data to the hourly level
#  and then I will train the model on the aggregated data.
daily_data = data.groupby("datetime", as_index=False).agg(
    total_count=("total_count", "sum"),
    count_pickups=("count_pickups", "sum"),
    count_rentals=("count_rentals", "sum"),
    temperature_c=("temperature_c", "mean"),
    perceived_temperature_c=("perceived_temperature_c", "mean"),
    humidity=("humidity", "mean"),
    windspeed_kmh=("windspeed_kmh", "mean"),
    conditions_clear=("conditions_clear", "mean"),
    conditions_clouds=("conditions_clouds", "mean"),
    conditions_heavy_rain=("conditions_heavy_rain", "mean"),
    conditions_light_rain=("conditions_light_rain", "mean"),
    is_holiday=("is_holiday", "max"),
)

daily_data = add_time_based_features(daily_data, col="datetime")
daily_data = add_lag_features(
    daily_data, target_col="total_count", lags=[1, 24, 168]
)
daily_data = add_rolling_features(
    daily_data, target_col="total_count", windows=[24, 168]
)

daily_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 32 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   datetime                      17379 non-null  datetime64[us]
 1   total_count                   17379 non-null  float64       
 2   count_pickups                 17379 non-null  float64       
 3   count_rentals                 17379 non-null  float64       
 4   temperature_c                 17379 non-null  float64       
 5   perceived_temperature_c       17379 non-null  float64       
 6   humidity                      17379 non-null  float64       
 7   windspeed_kmh                 17379 non-null  float64       
 8   conditions_clear              17379 non-null  float64       
 9   conditions_clouds             17379 non-null  float64       
 10  conditions_heavy_rain         17379 non-null  float64       
 11  conditions_light_rain         17379 non

In [190]:
daily_data.describe()

,datetime,total_count,count_pickups,count_rentals,temperature_c,perceived_temperature_c,humidity,windspeed_kmh,conditions_clear,conditions_clouds,...,is_weekend,hour_sin,hour_cos,total_count_lag_1,total_count_lag_24,total_count_lag_168,total_count_rolling_mean_24,total_count_rolling_std_24,total_count_rolling_mean_168,total_count_rolling_std_168
count,17379,17379.000000,17379.000000,17379.000000,17379.000000,17379.000000,17379.000000,17379.000000,17379.000000,17379.000000,...,17379.000000,17379.000000,1.737900e+04,17378.000000,17355.000000,17211.000000,17355.000000,17355.000000,17211.000000,17211.000000
mean,2012-01-02 15:41:22.858622,189.463088,35.676218,153.786869,15.354203,15.401116,62.722884,12.736233,0.656712,0.261465,...,0.288509,-0.004509,-3.345887e-03,189.471170,189.567848,190.583871,189.607635,154.690691,190.627635,158.146656
min,2011-01-01 00:00:00,1.000000,0.000000,0.000000,-7.100000,-16.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,-1.000000,-1.000000e+00,1.000000,1.000000,1.000000,18.041667,13.755565,49.904762,42.504687
25%,2011-07-04 22:30:00,40.000000,4.000000,34.000000,8.000000,6.000000,48.000000,7.000000,0.000000,0.000000,...,0.000000,-0.707107,-7.071068e-01,40.000000,40.000000,41.000000,133.250000,111.314685,140.767857,125.901662
50%,2012-01-02 21:00:00,142.000000,17.000000,115.000000,15.500000,16.000000,63.000000,13.000000,1.000000,0.000000,...,0.000000,0.000000,-1.836970e-16,142.000000,142.000000,144.000000,189.666667,154.505188,187.482143,150.296314
75%,2012-07-02 06:30:00,281.000000,48.000000,220.000000,23.000000,25.000000,78.000000,17.000000,1.000000,1.000000,...,1.000000,0.707107,7.071068e-01,281.000000,281.000000,282.000000,248.395833,201.344889,257.029762,210.007907
max,2012-12-31 23:00:00,977.000000,367.000000,886.000000,39.000000,50.000000,100.000000,57.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000e+00,977.000000,977.000000,977.000000,382.500000,294.511977,332.994048,266.183927
std,NaN,181.387599,49.305030,151.357286,9.049874,11.341858,19.292983,8.196891,0.474820,0.439445,...,0.453082,0.706814,7.074181e-01,181.389688,181.462851,181.772948,78.947826,62.612142,71.626899,55.364156


In [191]:
daily_data.shape

(17379, 32)

In [192]:
daily_data.datetime[0:5]

0   2011-01-01 00:00:00
1   2011-01-01 01:00:00
2   2011-01-01 02:00:00
3   2011-01-01 03:00:00
4   2011-01-01 04:00:00
Name: datetime, dtype: datetime64[us]

In [193]:
daily_data.columns

Index(['datetime', 'total_count', 'count_pickups', 'count_rentals',
       'temperature_c', 'perceived_temperature_c', 'humidity', 'windspeed_kmh',
       'conditions_clear', 'conditions_clouds', 'conditions_heavy_rain',
       'conditions_light_rain', 'is_holiday', 'dayofweek', 'year', 'month',
       'day', 'quarter', 'date', 'hour', 'is_month_start', 'is_month_end',
       'is_weekend', 'hour_sin', 'hour_cos', 'total_count_lag_1',
       'total_count_lag_24', 'total_count_lag_168',
       'total_count_rolling_mean_24', 'total_count_rolling_std_24',
       'total_count_rolling_mean_168', 'total_count_rolling_std_168'],
      dtype='str')

In [200]:
def train_and_evaluate_model(daily_data, feature_set_name):
    """Train and evaluate models using the specified feature set."""
    features = FEATURE_SETS[feature_set_name]
    daily_data.dropna(subset=features + [TARGET], inplace=True)
    X_train, y_train, X_test, y_test = splitting_data(daily_data, features)

    linear_model = linear_regression_model(X_train, y_train)
    rf_model = random_forest_model(X_train, y_train)
    xgb_model = xgboost_model(X_train, y_train)

    errors = {
        "Linear Regression": metrics(linear_model, X_test, y_test),
        "Random Forest": metrics(rf_model, X_test, y_test),
        "XGBoost": metrics(xgb_model, X_test, y_test),
    }
    configs = {
        "feature_set": feature_set_name,
        "features": features,
        "X_train_shape": X_train.shape,
        "y_train_shape": y_train.shape,
        "X_test_shape": X_test.shape,
        "y_test_shape": y_test.shape,
    }
    return errors, configs

In [197]:
op_dict = {}
for feature_set_name in FEATURE_SETS.keys():
    errors, configs = train_and_evaluate_model(daily_data, feature_set_name)
    op_dict[feature_set_name] = {"errors": errors, "configs": configs}

all_feature_set = set()
for feature_set_name in FEATURE_SETS.keys():
    all_feature_set.update(FEATURE_SETS[feature_set_name])
FEATURE_SETS["all_features"] = list(all_feature_set)

errors, configs = train_and_evaluate_model(daily_data, "all_features")
op_dict["all_features"] = {"errors": errors, "configs": configs}

In [199]:
for feature_set_name, results in op_dict.items():
    print(f"Feature Set: {feature_set_name}")
    print("Errors:")
    for model_name, error_metrics in results["errors"].items():
        print(
            f"  {model_name}: MAE={error_metrics['mae']:.2f},\
               RMSE={error_metrics['rmse']:.2f}"
        )
    print("Configs:")
    for config_key, config_value in results["configs"].items():
        print(f"  {config_key}: {config_value}")
    print("\n")

Feature Set: weather_only
Errors:
  Linear Regression: MAE=150.23,               RMSE=208.11
  Random Forest: MAE=161.82,               RMSE=219.07
  XGBoost: MAE=149.59,               RMSE=206.76
Configs:
  feature_set: weather_only
  features: ['temperature_c', 'perceived_temperature_c', 'humidity', 'windspeed_kmh', 'conditions_clear', 'conditions_clouds', 'conditions_heavy_rain', 'conditions_light_rain']
  X_train_shape: (12047, 8)
  y_train_shape: (12047,)
  X_test_shape: (5164, 8)
  y_test_shape: (5164,)


Feature Set: feature_set_1
Errors:
  Linear Regression: MAE=150.11,               RMSE=207.99
  Random Forest: MAE=161.66,               RMSE=218.78
  XGBoost: MAE=149.36,               RMSE=206.68
Configs:
  feature_set: feature_set_1
  features: ['is_month_start', 'is_month_end', 'temperature_c', 'perceived_temperature_c', 'humidity', 'windspeed_kmh', 'conditions_clear', 'conditions_clouds', 'conditions_heavy_rain', 'conditions_light_rain', 'is_holiday']
  X_train_shape: (1204